# 《PythAPCS123》單元 13-6：時間超限（TLE）診斷：運算量估算、無窮迴圈與隱形效能坑洞防制

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/johnnyy-lab/APCS1to3/blob/main/PythAPCS123_13-6_time_limit_exceeded_and_performance_tuning.ipynb)

**適合對象**：程式設計初學者（完全零基礎） / APCS 扎根學習者
**核心目標**：攻克程式競賽中令無數考生抓狂的「超時噩夢」——時間超限（Time Limit Exceeded, TLE）！建立 APCS 伺服器「1 秒運行時限」與「$10^7$ 基本操作次數」的實戰直覺換算模型，學會審視資料規模 $N$ 與大 $O$ 複雜度以提前判斷演算法能否過關。深入排查無法收斂的死迴圈，地毯式揪出初學者最常掉入的三大隱形效能黑洞：迴圈內的 `list in` 線性搜尋、串列頭部頻繁插入刪除（`insert(0)` / `pop(0)`）、以及字串反覆 `+=` 拼接的記憶體搬移開銷。最後解鎖競賽終極極速 I/O 神器 `sys.stdin.readline`，徹底告別 TLE！


### 13.6.1 什麼是 TLE？APCS 伺服器 1 秒限制與 Python 運算速度極限

在 APCS 實作題或各類線上評判系統（如 ZeroJudge、LeetCode）中，每一道題目都會在題目的規格說明處明確標示一個極其關鍵的限制：**「時間限制（Time Limit）：1.0 秒」**。

評判系統的運作方式是：後端伺服器在啟動你的 Python 程式時，會同時啟動一個精密的硬體計時器。如果程式在 1.000 秒之內計算完畢並順利輸出結果，評判系統才會比對答案；反之，一旦計時器抵達 1.0 秒而程式仍在迴圈中狂奔，伺服器就會毫不留情地發送作業系統強制中斷訊號（`SIGKILL`），直接將你的程式殺死，並在螢幕上記下一記刺眼的 **`TLE（Time Limit Exceeded）`**，該測試點直接計 0 分！

那麼，在 1.0 秒的時間內，Python 到底能跑幾次運算？
由於 Python 是高階動態直譯語言，帶有動態型別檢查與直譯器開銷，因此在競賽伺服器的常見硬體規格下：
- C/C++ 語言 1 秒約可跑 $10^8$ 次基本運算（約一億次）。
- **Python 語言 1 秒的安全運算量約為 $10^7$ 次基本操作（約一千萬次）**。

這條「**$10^7$ 操作次數安全黃金法則**」，是所有 APCS 考生在動筆寫下第一行代碼之前，必須刻在腦海中的最高軍規！


In [ ]:
# 13.6.1 程式碼演示：實測 Python 執行一千萬次（10^7）基本迴圈的耗時
import time

print("=== 實測 Python 10^7 次基本運算速度 ===")
N = 10**7 # 一千萬次

start_time = time.time()
# 執行最簡單的純迴圈遞增計數
counter = 0
for _ in range(N):
    counter += 1
end_time = time.time()

elapsed = end_time - start_time
print(f"迴圈總次數: {N:,} 次")
print(f"實際消耗時間: {elapsed:.3f} 秒")

if elapsed <= 1.0:
    print("✅ 結論：10^7 次純運算能夠在 1.0 秒安全限制內順利完成！")
else:
    print("⚠️ 提示：稍微超過 1 秒，若迴圈內有複雜操作，次數需進一步收縮至 10^6 次！")


### 13.6.1 語法重點回顧與核心觀念提煉

時間限制與運算量評估的兩大心智模型：
1. **$10^7$ 運算紅線**：如果你設計的演算法，最差情況下的運算總次數會超過 $10^7$ 次，那麼不要懷疑，上傳到 OJ 後「百分之百會拿到 TLE」！此時不要試圖做無謂的微調，必須在紙上重新設計更高效率的演算法。
2. **動態複雜度自覺**：迴圈內每多寫一行程式碼（特別是呼叫內建函式），迴圈單次開銷就會成倍增加。因此，若迴圈內有複雜的字串或容器操作，安全次數上限應下調至 $10^6$ 次左右。


In [ ]:
# 13.6.1 學生實作練習：運算量安全估算器
# 任務說明：實作 is_operation_count_safe(total_operations) 函式
# 根據 APCS 1 秒限制黃金法則（安全上限為 10^7 = 10,000,000 次操作）
# 若預估總運算次數 total_operations <= 10^7，回傳 True（安全）
# 若超過 10^7，回傳 False（高度面臨 TLE 風險）

def is_operation_count_safe(total_operations: int) -> bool:
    # 請在此處完成判斷
    SAFE_LIMIT = 10**7
    return total_operations <= SAFE_LIMIT

# 測試用例
print("五百萬次運算是否安全:", is_operation_count_safe(5_000_000))
print("一億次運算是否安全:", is_operation_count_safe(100_000_000))


In [ ]:
# 13.6.1 單元測試驗證
assert is_operation_count_safe(10**7) == True
assert is_operation_count_safe(10**7 - 1) == True
assert is_operation_count_safe(10**7 + 1) == False
assert is_operation_count_safe(10**9) == False
print("13.6.1 單元測試全數通過！")


### 13.6.2 複雜度量級換算表：在 N=10^5 時，O(N) 與 O(N^2) 的生與死

拿到 APCS 題目時，高手與新手的最大差異在於：**高手第一眼看的不是題目故事，而是「測資範圍限制（Constraints）」中的 $N$！**

常見測資規模與容許時間複雜度對照表：
- **若 $N \le 10$**：容許 $O(N!)$ 暴力全排列或 $O(2^N)$ 窮舉。
- **若 $N \le 10^3$**：容許 $O(N^2)$ 雙層迴圈（$N^2 = 10^6 \le 10^7$，安全秒過！）。
- **若 $N \le 10^5$（APCS 經典題型常見規模）**：
  - 若寫 $O(N)$ 線性掃描或雙指標：運算次數 $10^5$，僅耗時約 0.01 秒，**穩穩 AC**！
  - 若寫 $O(N \log N)$ 排序：運算次數約 $10^5 \times 17 \approx 1.7 \times 10^6$，耗時約 0.1 秒，**順利 AC**！
  - 若寫 $O(N^2)$ 雙層迴圈：運算次數高達 $(10^5)^2 = 10^{10}$ 次！在 Python 裡需要跑整整 **1000 秒（超過 16 分鐘）**！在 1 秒的裁判系統上必死無疑！

學會由 $N$ 的大小反推演算法上限，是避免浪費寶貴考試時間寫出必定 TLE 代碼的最強前瞻雷達。


In [ ]:
# 13.6.2 程式碼演示：小規模 vs 大規模下 O(N) 與 O(N^2) 耗時爆炸對比
import time

# 案例：檢查整數串列中是否存在任兩數相加等於目標值 (Two Sum)
def naive_two_sum_n2(arr, target):
    # O(N^2) 雙層迴圈暴力窮舉
    n = len(arr)
    for i in range(n):
        for j in range(i + 1, n):
            if arr[i] + arr[j] == target:
                return (i, j)
    return None

def fast_two_sum_on(arr, target):
    # O(N) 雜湊集合/字典快速查詢
    seen = {}
    for i, val in enumerate(arr):
        comp = target - val
        if comp in seen:
            return (seen[comp], i)
        seen[val] = i
    return None

# 測試規模 N = 5000
test_size = 5000
test_data = list(range(test_size))
test_target = -1 # 故意設為不存在，測試最差情況

print(f"--- 測資規模 N = {test_size} 最差情況耗時測試 ---")
t0 = time.time()
fast_two_sum_on(test_data, test_target)
t_fast = time.time() - t0
print(f"O(N) 雜湊解法耗時: {t_fast:.5f} 秒 ⚡")

t0 = time.time()
naive_two_sum_n2(test_data, test_target)
t_slow = time.time() - t0
print(f"O(N^2) 暴力解法耗時: {t_slow:.3f} 秒 🐢")
print(f"速度差距: {t_slow / max(t_fast, 1e-6):.1f} 倍！若 N 放大到 10^5，O(N^2) 將徹底癱瘓！")


### 13.6.2 語法重點回顧與核心觀念提煉

考場依 $N$ 定策的心法法則：
1. **$N \ge 10^5$ 鐵律**：題目只要出現 $N \ge 10^5$，所有巢狀雙層迴圈（`for i ...: for j ...:`）全部禁止使用！你的演算法必須維持在 $O(N)$ 或 $O(N \log N)$ 之內。
2. **空間換時間（Space-Time Tradeoff）**：雙層迴圈往往可以透過「集合 `set`」或「字典 `dict`」將內層查找降為 $O(1)$，或者透過「先排序 $O(N \log N)$ 再雙指標 $O(N)$」來化解 $O(N^2)$ 的危機。


In [ ]:
# 13.6.2 學生實作練習：演算法可行性評估器
# 任務說明：實作 evaluate_algorithm_feasibility(n, complexity_type) 函式
# 傳入資料規模 n，以及演算法複雜度類型字串：
# - "O(N)": 總次數為 n
# - "O(NlogN)": 總次數估算為 n * 20
# - "O(N^2)": 總次數為 n ** 2
# 判斷總運算次數是否小於等於 10^7（1 秒安全上限）
# 若在限制內回傳 "PASS"，若超過回傳 "TLE"！

def evaluate_algorithm_feasibility(n: int, complexity_type: str) -> str:
    # 請在此處計算預估次數並比對
    ops = 0
    if complexity_type == "O(N)":
        ops = n
    elif complexity_type == "O(NlogN)":
        ops = n * 20
    elif complexity_type == "O(N^2)":
        ops = n ** 2
        
    return "PASS" if ops <= 10**7 else "TLE"

# 測試用例
print("N=10^5 跑 O(N):", evaluate_algorithm_feasibility(10**5, "O(N)"))
print("N=10^5 跑 O(N^2):", evaluate_algorithm_feasibility(10**5, "O(N^2)"))
print("N=1000 跑 O(N^2):", evaluate_algorithm_feasibility(1000, "O(N^2)"))


In [ ]:
# 13.6.2 單元測試驗證
assert evaluate_algorithm_feasibility(10**5, "O(N)") == "PASS"
assert evaluate_algorithm_feasibility(10**5, "O(NlogN)") == "PASS"
assert evaluate_algorithm_feasibility(10**5, "O(N^2)") == "TLE"
assert evaluate_algorithm_feasibility(1000, "O(N^2)") == "PASS"
print("13.6.2 單元測試全數通過！")


### 13.6.3 致命無窮迴圈診斷：while 終止條件無法收斂

除了演算法複雜度過高之外，導致 TLE 的另一大元兇是：**程式碼陷入了永遠無法跳出的「無窮迴圈（Infinite Loop）」**！

在 APCS 考場上，初學者最常犯的無窮迴圈有兩大模式：
1. **計數器忘記更新**：
   在撰寫 `while i < n:` 時，迴圈內部專注處理業務邏輯，卻在最後「漏寫了 `i += 1`」！導致變數 `i` 永遠維持原值，條件恆為 `True`，程式像倉鼠跑滾輪一樣瘋狂空轉直到 1 秒時限被伺服器斬首。
2. **分支路徑中漏掉推進**：
   當迴圈內部包含多個 `if-elif-else` 分支時，某個分支內部寫了 `continue`，但推進計數器 `i += 1` 卻寫在 `continue` 的下方！導致一旦走進該分支，每次都 `continue` 跳過推進，再次引爆無窮迴圈。
3. **收斂條件永不滿足（浮點數或符號跳躍）**：
   例如寫 `while x != 0:`，但在迴圈內部每次執行 `x -= 2`，若輸入的 `x` 是奇數，數值會從 $3 \to 1 \to -1 \to -3$，永遠跨越 0 而無法停機！

防禦無窮迴圈的最高準則：**只要使用 `while`，第一件事必須檢查「每一次迭代，狀態是否都嚴格朝著終止條件前進一步？」**


In [ ]:
# 13.6.3 程式碼演示：計數器跳躍陷阱與步數安全閥防禦
def dangerous_step(start_x):
    # 潛在無窮迴圈：若 start_x 為奇數，x -= 2 永遠碰不到 0！
    x = start_x
    print(f"\n[測試 dangerous_step({start_x})]")
    
    # 加上「步數安全閥（Step Limit Guard）」防範在演示中卡死
    MAX_STEPS = 10
    steps = 0
    
    while x != 0:
        steps += 1
        if steps > MAX_STEPS:
            print("  ⚠️ 警告：超過 10 步未收斂，已由安全閥強制中斷！偵測到無窮迴圈！")
            return "INFINITE_LOOP"
        x -= 2
        
    print(f"  ✅ 順利收斂，最終 x = {x}")
    return "CONVERGED"

# 偶數能收斂 (4 -> 2 -> 0)
dangerous_step(4)

# 奇數跨越 0 陷入死迴圈 (3 -> 1 -> -1 -> -3 ...)
dangerous_step(3)


### 13.6.3 語法重點回顧與核心觀念提煉

診斷與預防無窮迴圈的自查清單：
1. **計數器位置檢查**：確保 `i += 1` 絕對不會被任何 `continue` 提早跳過。
2. **終止條件改用不等式**：避免寫 `while x != 0:`，改用 `while x > 0:`！這樣即便步長為 2 或出現跳躍，一旦數值小於等於 0 也會立刻安全停機。
3. **除錯 print 標記**：當本機執行卡住時，在 while 迴圈第一行加入 `print(f"[DEBUG] i={i}, x={x}")`，觀察數值是否有在規律遞增或收斂。


In [ ]:
# 13.6.3 學生實作練習：安全除法模擬器
# 任務說明：實作 safe_count_divisions(n, divisor) 函式
# 計算正整數 n 在大於 1 的前提下，可以連續被 divisor（大於 1）整除幾次
# 防禦要求：
# 1. 確保 while 迴圈每次迭代皆能嚴格縮小 n 的數值
# 2. 終止條件嚴密，若 n 已經無法被整除則必須即刻中斷，杜絕死迴圈！
# 回傳連續整除的總次數！

def safe_count_divisions(n: int, divisor: int) -> int:
    count = 0
    # 請在此處實作安全收斂的 while 迴圈
    while n > 1 and (n % divisor == 0):
        n //= divisor
        count += 1
    return count

# 測試用例
print("24 被 2 連續整除次數:", safe_count_divisions(24, 2)) # 24 -> 12 -> 6 -> 3 (3次)
print("100 被 5 連續整除次數:", safe_count_divisions(100, 5)) # 100 -> 20 -> 4 (2次)


In [ ]:
# 13.6.3 單元測試驗證
assert safe_count_divisions(24, 2) == 3
assert safe_count_divisions(100, 5) == 2
assert safe_count_divisions(7, 2) == 0
assert safe_count_divisions(16, 2) == 4
print("13.6.3 單元測試全數通過！")


### 13.6.4 隱形效能地雷一：在迴圈內反覆執行 O(N) 操作（list in, insert, pop）

在 APCS 考場上，許多學生看著自己的程式碼自信滿滿：「我只寫了一層 `for` 迴圈呀！複雜度不就是 $O(N)$ 嗎？為什麼還會 TLE？」

這正是初學者最容易踩入的**「隱形 $O(N^2)$ 效能地雷」**！
雖然表面上只有一層 `for` 迴圈，但如果迴圈「內部」呼叫了某個底層也是 $O(N)$ 的操作，兩者相乘起來，整體複雜度瞬間暴增為 $O(N \times N) = O(N^2)$！

考場最兇殘的三大隱形操作：
1. **在串列中使用 `in` 關鍵字**：
   串列（`list`）的 `x in my_list` 是**線性搜尋**，必須從頭比對到尾，單次耗時 $O(N)$！若在長度為 $N$ 的迴圈中寫 `if x in my_list:`，就是標準的 $O(N^2)$ 死亡陷阱！
   - *正解*：將容器轉換為集合 `set` 或字典 `dict`，其 `in` 查詢是基於雜湊表，單次僅需 **$O(1)$ 常數時間**！
2. **在串列開頭插入或刪除元素**：
   `my_list.insert(0, x)` 或 `my_list.pop(0)`，因為底層陣列必須把後續所有元素全部往前或往後搬移一格，單次耗時 $O(N)$！
   - *正解*：尾端操作 `append()` 與 `pop()` 為 $O(1)$；若確實需要雙向佇列，使用 `collections.deque`。


In [ ]:
# 13.6.4 程式碼演示：list in vs set in 效能天壤之別
import time

N = 50000
query_list = list(range(N))
query_set = set(query_list) # 轉為集合只需一次 O(N)

test_queries = [N - 1] * 2000 # 重複查詢 2000 次

# 測試 A: 串列線性查找 O(N) 重複執行
t0 = time.time()
found_count = 0
for q in test_queries:
    if q in query_list: # 每次查找需花費 O(N)
        found_count += 1
t_list = time.time() - t0

# 測試 B: 集合雜湊查找 O(1) 重複執行
t0 = time.time()
found_count = 0
for q in test_queries:
    if q in query_set: # 每次查找僅需 O(1)
        found_count += 1
t_set = time.time() - t0

print(f"長度 N = {N:,} 查詢 2000 次耗時對比:")
print(f"  使用 list in 耗時: {t_list:.3f} 秒 🐢 (極易引發 TLE)")
print(f"  使用 set in 耗時: {t_set:.5f} 秒 ⚡ (極速秒過)")
print(f"效能差距: {t_list / max(t_set, 1e-6):.1f} 倍！")


### 13.6.4 語法重點回顧與核心觀念提煉

操作複雜度防禦對照表：
| 常見操作 | 串列 `list` | 集合 `set` / 字典 `dict` | 佇列 `collections.deque` |
| :--- | :---: | :---: | :---: |
| **成員查找 (`x in C`)** | $O(N)$ ❌ **極度危險** | $O(1)$ ✅ **推薦必用** | $O(N)$ |
| **尾端加入 (`append`)** | $O(1)$ ✅ | $O(1)$（add） | $O(1)$ ✅ |
| **開頭插入 (`insert(0)`)** | $O(N)$ ❌ **避免使用** | 不支援（無序） | $O(1)$（appendleft）✅ |
| **開頭取出 (`pop(0)`)** | $O(N)$ ❌ **避免使用** | 不支援（無序） | $O(1)$（popleft）✅ |

**考場口訣**：「**頻繁查詢用 set，首尾操作用 deque，不要拿 list 當佇列！**」


In [ ]:
# 13.6.4 學生實作練習：極速清單去重與交集統計器
# 任務說明：實作 count_common_elements_fast(list_a, list_b) 函式
# 計算同時存在於 list_a 與 list_b 的不重複元素個數
# 嚴格效能要求：必須利用 set 將查找或交集複雜度降至 O(N)，
# 絕對不可使用雙層迴圈或在迴圈內使用 x in list_b！

def count_common_elements_fast(list_a: list, list_b: list) -> int:
    # 請利用 set 實現極速交集或查找
    set_a = set(list_a)
    set_b = set(list_b)
    return len(set_a & set_b)

# 測試用例
print("共同元素個數:", count_common_elements_fast([1, 2, 2, 3, 4], [2, 3, 5]))


In [ ]:
# 13.6.4 單元測試驗證
assert count_common_elements_fast([1, 2, 3], [2, 3, 4]) == 2
assert count_common_elements_fast([1, 1, 1], [1, 1]) == 1
assert count_common_elements_fast([1, 2], [3, 4]) == 0
assert count_common_elements_fast([], [1, 2]) == 0
print("13.6.4 單元測試全數通過！")


### 13.6.5 隱形效能地雷二：迴圈內大量字串 + 串接

在處理字串題或文字模擬題時，初學者常習慣使用加號 `+=` 來將字元一個個拼接到字串後面：
```python
# 致命寫法：在大規模迴圈中反覆 += 拼接字串
result = ""
for char in large_text:
    result += char
```

在 Python 中，字串（`str`）是**「不可變物件（Immutable Object）」**。這意味著：一旦一個字串被建立，它在記憶體中的內容就永遠無法被就地修改。
當你執行 `result += char` 時，Python 背後實際上做了以下昂貴的動作：
1. 在記憶體中重新申請一塊「長度為當前長度 + 1」的全新記憶體空間。
2. 將舊 `result` 中的所有字元「一個個完整拷貝複製」到新空間中。
3. 把新字元塞到末尾，並銷毀舊字串。

如果迴圈執行 $N$ 次，每次搬移的字元數分別為 $1, 2, 3, \dots, N-1$。總複製次數高達 $\frac{N(N-1)}{2} \approx \frac{N^2}{2}$ 次！
這直接把一個原本純線性的任務，硬生生拉進了 $O(N^2)$ 的地獄！

**唯一救星：`''.join(list)`**！
將字元先用可變串列的 `append()` 收集起來（均攤 $O(1)$），最後用 `''.join()` 一次性向作業系統申請足額記憶體合併，全流程僅需 $O(N)$！


In [ ]:
# 13.6.5 程式碼演示：字串 += 累加 vs list.append + join 效能慘烈對比
import time

N = 50000

# 測試 A: 使用 += 進行字串累積 (O(N^2))
t0 = time.time()
s_bad = ""
for i in range(N):
    s_bad += "x"
t_plus = time.time() - t0

# 測試 B: 使用 list 收集後一次 join (O(N))
t0 = time.time()
chars = []
for i in range(N):
    chars.append("x")
s_good = "".join(chars)
t_join = time.time() - t0

print(f"拼接 {N:,} 個字元耗時測試:")
print(f"  字串 += 耗時: {t_plus:.4f} 秒 🐢 (記憶體反覆搬移，若 N=10^6 將極度緩慢)")
print(f"  join() 耗時: {t_join:.5f} 秒 ⚡ (一次性建立，極速流暢)")
print(f"效能差距: {t_plus / max(t_join, 1e-6):.1f} 倍！")


### 13.6.5 語法重點回顧與核心觀念提煉

字串構建的黃金法則：
1. **迴圈內字串禁止 `+=`**：只要遇到「需要在迴圈內逐步拼裝超長字串」的場景，大腦要立刻反射：
   ```python
   # 標準高效模板：
   pieces = []
   for item in generator:
       pieces.append(str(item))
   ans = "".join(pieces) # 或 " ".join(pieces)
   ```
2. **列表生成式更加精練**：
   ```python
   ans = "".join(str(x) for x in arr)
   ```
一念之差，直接將原本可能 TLE 的代碼瞬間提速百倍以上！


In [ ]:
# 13.6.5 學生實作練習：高效交替字串產生器
# 任務說明：實作 build_alternating_string(n) 函式
# 產生長度為 n 的 'A', 'B' 交替字串（如 n=5 產生 "ABABA"，n=4 產生 "ABAB"）
# 嚴格效能要求：必須使用串列 append 搭配 ''.join() 完成組裝，
# 絕不可使用字串 += 逐字累加！

def build_alternating_string(n: int) -> str:
    # 請在此處使用 list 搭配 join 組裝字串
    chars = []
    for i in range(n):
        chars.append('A' if i % 2 == 0 else 'B')
    return "".join(chars)

# 測試用例
print("長度 5 交替字串:", build_alternating_string(5))
print("長度 4 交替字串:", build_alternating_string(4))


In [ ]:
# 13.6.5 單元測試驗證
assert build_alternating_string(5) == "ABABA"
assert build_alternating_string(4) == "ABAB"
assert build_alternating_string(1) == "A"
assert build_alternating_string(0) == ""
print("13.6.5 單元測試全數通過！")


### 13.6.6 考場極速 I/O 秘密武器：大量輸入資料時以 sys.stdin.readline 取代 input()

當你把演算法複雜度壓到了最優的 $O(N)$，避開了所有的隱形地雷，但在上傳 APCS 遇到某些「超巨大測資（例如有 50 萬行整數需要讀入）」時，程式依然可能在讀取階段就超時！

原因在於 Python 內建的 `input()` 函式內部做了許多額外的安全處理（包括去除行尾換行符號、處理終端機互動提示詞等等）。當面對數十萬行輸入時，單單是 `input()` 本身就可能耗掉 0.8 到 1.5 秒！

為了解決這個 I/O 瓶頸，競技程式選手擁有一件終極秘密武器：**`sys.stdin.readline`**！
直接向作業系統的標準輸入緩衝區索取一整行文字，速度比 `input()` 快上 **3 到 5 倍**！

使用 `sys.stdin.readline` 的兩大關鍵細節：
1. **行末換行符號殘留**：`sys.stdin.readline()` 會保留行尾的換行符號 `\n`，因此若需要純字串，請加上 `.strip()` 或 `.rstrip('\n')`。
2. **極速別名慣用法（考場首選）**：在程式碼最開頭寫下一行別名替換，後續的程式碼完全不需要改動：
   ```python
   import sys
   input = sys.stdin.readline # 神來一筆，全局瞬間加速！
   ```


In [ ]:
# 13.6.6 程式碼演示：模擬百萬字元串流讀取 input() vs sys.stdin.readline
import io
import sys
import time

# 建立 20,000 行數字的輸入串流
N_LINES = 20000
raw_text = "\n".join(str(i) for i in range(N_LINES)) + "\n"

# 測試 A: 使用標準 input()
sys.stdin = io.StringIO(raw_text)
t0 = time.time()
lines_a = []
for _ in range(N_LINES):
    lines_a.append(input())
t_input = time.time() - t0

# 測試 B: 使用 sys.stdin.readline
sys.stdin = io.StringIO(raw_text)
t0 = time.time()
lines_b = []
for _ in range(N_LINES):
    lines_b.append(sys.stdin.readline().rstrip('\n'))
t_readline = time.time() - t0

print(f"讀取 {N_LINES:,} 行資料耗時對比:")
print(f"  標準 input() 耗時: {t_input:.4f} 秒 🐢")
print(f"  sys.stdin.readline 耗時: {t_readline:.4f} 秒 ⚡")
print(f"I/O 速度提升約: {t_input / max(t_readline, 1e-6):.1f} 倍！")


### 13.6.6 語法重點回顧與核心觀念提煉

考場極速 I/O 最佳實踐準則：
1. **巨量輸入模板**：只要題目輸入行數大於 $10^4$，養成在第一行引入極速 I/O 的好習慣：
   ```python
   import sys
   input = sys.stdin.readline
   ```
2. **快速解析多整數**：結合 `map` 與 `split`：
   ```python
   # 讀取一行兩個整數：
   n, m = map(int, input().split())
   # 讀取一行整數串列：
   arr = list(map(int, input().split()))
   ```
掌握此技，便能徹底清除最後一哩路的 I/O 延遲，以最充裕的時間餘裕通過所有超大測資點！


In [ ]:
# 13.6.6 學生實作練習：模擬極速串流解析器
# 任務說明：實作 parse_stream_sum(stream_str) 函式
# 傳入包含多行整數的 stream_str 字串（每行一個整數）
# 利用 io.StringIO 與 sys.stdin.readline 進行讀取
# 計算所有整數的總和並回傳！

import io
import sys

def parse_stream_sum(stream_str: str) -> int:
    old_stdin = sys.stdin
    sys.stdin = io.StringIO(stream_str)
    
    total = 0
    # 請在此處使用 sys.stdin.readline() 完成快速讀取
    while True:
        line = sys.stdin.readline()
        if not line: # 讀至 EOF
            break
        text = line.strip()
        if text:
            total += int(text)
            
    sys.stdin = old_stdin
    return total

# 測試用例
sample_stream = "10\n20\n30\n"
print("串流總和:", parse_stream_sum(sample_stream))


In [ ]:
# 13.6.6 單元測試驗證
assert parse_stream_sum("1\n2\n3\n4\n5\n") == 15
assert parse_stream_sum("100\n-50\n25\n") == 75
assert parse_stream_sum("") == 0
assert parse_stream_sum("42\n") == 42
print("13.6.6 單元測試全數通過！")


## 13.6 總結與 TLE 效能排查全圖譜

在本單元中，我們建立了一整套從量級估算到代碼微調的 TLE 終極防禦體系。遇到超時評判時，請依據下表循序排查：

| TLE 潛在元兇 | 觸發原因與量級災難 | 考場極速根治手段 |
| :--- | :--- | :--- |
| **演算法複雜度過高** | $N=10^5$ 卻寫了 $O(N^2)$ 雙迴圈（超時 1000 倍） | 看 $N$ 定策：$N \ge 10^5$ 必須用 $O(N)$ 雙指標或 $O(N \log N)$ 排序 |
| **無窮迴圈（死迴圈）** | while 計數器未更新、被 continue 跳過 | 終止條件改用不等式 `>`，確認每次循環皆有推進 |
| **迴圈內 `x in list`** | 串列線性查找為 $O(N)$，外加迴圈成 $O(N^2)$ | 將查找容器換成**集合 `set`**，單次查找瞬間降為 $O(1)$ |
| **串列開頭 `insert/pop(0)`** | 底層記憶體每次全員位移 $O(N)$ | 尾端操作或引入 `collections.deque`（雙向佇列） |
| **迴圈內字串 `+=`** | 字串不可變，反覆複製搬移整串記憶體 | 一律先用串列 `append()` 收集，最後用 `''.join()` 一次拼裝 |
| **大量測資 I/O 延遲** | 數十萬行 `input()` 消耗過多時間 | 開頭加入 `import sys; input = sys.stdin.readline` |

### 🚀 下一步學習指引
在攻克了 CE（語法）、RE（崩潰）、WA（邏輯）與 TLE（效能）之後，許多考生還常遭遇一種最令人扼腕的冤枉情境：**「邏輯完全正確、答案算得分毫不差、效能也極度飛快，但上傳 OJ 依然拿到 WA！」**
這通常是因為「輸出格式（Presentation Error / Format）」出現了肉眼難以察覺的微小瑕疵——例如多印了一個空格、少換了一行、或者題目要整數 `4` 卻印成了浮點數 `4.0`。
在下一單元 **13-7《輸出格式防禦與對齊心法（Presentation / Format WA 防範）》** 中，我們將專注鍛鍊毫釐不差的字元級輸出對齊神技！
